# 在 Colab 用 webcam 跑 YOLO（瀏覽器鏡頭版）

`yolo_only.ipynb` 第 4 格是**純文字說明**，點了不會有反應。
原因：**Colab 的 VM 在 Google 雲端，那台機器沒有實體相機。**

```python
cap = cv2.VideoCapture(0)
cap.isOpened()      # → False，永遠
```

要用 webcam，只有兩條路：

| 方案 | 在哪跑 | 說明 |
|---|---|---|
| **A. 回自己的電腦**（推薦） | 本機終端機 | 真即時、30 FPS，考試就是這樣 |
| **B. 這本 notebook** | Colab 瀏覽器 | 用 JavaScript 抓**你瀏覽器**的鏡頭 |

## ⚠️ 為什麼全部擠在一格

Colab 的 `eval_js` 綁定「**當前 cell**」的輸出 iframe
（`google/colab/output/_js.py` 用 `parent=kernel.shell.parent_header` 送出請求）。
**每格是獨立的 `window`**，所以在第 2 格設的 `window.__ivdc`，第 3 格完全看不到 ——
這正是錯誤 `camera not started - run the init cell first` 的成因。

因此**開鏡頭 + 抓幀 + 偵測全部在同一格**，不要拆。

## 前提

- ✅ **Chrome / Edge**（Safari、手機通常不行）
- ✅ 網址是 `colab.research.google.com`
- ✅ 授權時按「**允許**」
- ⚠️ 這是**逐張抓拍**，不是即時串流；**別拿這裡的 FPS 當效能指標**


## 1. 安裝


In [ ]:
%pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)


## 2. 開鏡頭 + 抓 10 幀跑 YOLO（**整段同一格，不要拆**）

跑完會看到：每次抓幀的尺寸/亮度、10 幀的偵測結果、以及最後一幀的標註圖。

**若彈出鏡頭授權，請按「允許」。**


In [ ]:
from IPython.display import display, Image
from google.colab.output import eval_js
from base64 import b64decode
import json
import cv2
import numpy as np
from ultralytics import YOLO

# Everything lives in ONE cell on purpose: eval_js evaluates "within the
# context of the current cell", and each cell has its own output iframe.
# `window.__ivdc` set in another cell would be invisible here.

CAPTURE_JS = """
(async () => {
  const S = window.__ivdc = window.__ivdc || {};

  if (!S.video) {
    const div = document.createElement('div');
    const video = document.createElement('video');
    video.style.display = 'block';
    video.width = 640;
    video.setAttribute('playsinline', '');
    video.muted = true;
    div.appendChild(video);
    document.body.appendChild(div);
    S.div = div;
    S.video = video;
    S.canvas = document.createElement('canvas');
  }

  if (!S.stream) {
    S.stream = await navigator.mediaDevices.getUserMedia({video: true});
    S.video.srcObject = S.stream;
    await S.video.play();
  }

  // wait for the first real frame
  for (let i = 0; i < 50 && !(S.video.videoWidth > 0); i++) {
    await new Promise(r => setTimeout(r, 100));
  }
  if (!(S.video.videoWidth > 0)) {
    throw new Error('camera opened but no frames arrived - close other apps using the camera and retry');
  }

  S.canvas.width = S.video.videoWidth;
  S.canvas.height = S.video.videoHeight;
  S.canvas.getContext('2d').drawImage(S.video, 0, 0);
  return JSON.stringify({
    w: S.video.videoWidth,
    h: S.video.videoHeight,
    data: S.canvas.toDataURL('image/jpeg', 0.9),
  });
})()
"""


def grab_frame():
    """Grab one frame from the browser camera and return a BGR ndarray."""
    payload = json.loads(eval_js(CAPTURE_JS))
    binary = b64decode(payload["data"].split(",")[1])
    frame = cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)
    print(f"  captured {payload['w']}x{payload['h']}, brightness {frame.mean():.1f}")
    return frame


model = YOLO("yolov8n.pt")   # ~6 MB download on first use

N = 10
last_result = None
for i in range(N):
    try:
        frame = grab_frame()
    except Exception as exc:
        print(f"frame {i} failed -> {type(exc).__name__}: {exc}")
        if i == 0:
            print()
            print("排查：")
            print("  permission / NotAllowedError -> 點網址列鎖頭圖示，相機設為「允許」，重跑本格")
            print("  no frames arrived            -> 關掉 Teams / Zoom / 相機 App 再試")
            print("  mediaDevices is undefined    -> 必須是 colab.research.google.com + Chrome/Edge")
        break

    r = model.predict(frame, imgsz=640, conf=0.25, verbose=False)[0]
    n = len(r.boxes)
    top = float(r.boxes.conf.max()) if n else 0.0
    print(f"frame {i:02d}: {n} 個物件, 最高信心 {top:.2f}, 推論 {r.speed['inference']:.0f} ms")
    last_result = r

if last_result is not None:
    last_result.save(filename="colab_webcam_last.jpg")
    print("\n最後一幀 -> colab_webcam_last.jpg")
    display(Image("colab_webcam_last.jpg", width=640))


## 3. 沒偵測到東西？

**幾乎都不是程式的問題：**

- COCO 模型只認 **80 類真實物體**（person / cup / bottle / cell phone / laptop / keyboard…）
- 對著牆壁、天花板、純色 → 0 個很正常
- 光線太暗 → 先開燈，或把 `conf=0.25` 降到 `0.15`

**拿這些給它看**：杯子、瓶子、手機、鍵盤、自己的臉（person）。

## 4. 用完記得關鏡頭

鏡頭指示燈不會自己熄。在 Colab 跑這格關閉串流：

```python
eval_js('''
(() => {
  const S = window.__ivdc;
  if (S && S.stream) { S.stream.getTracks().forEach(t => t.stop()); S.stream = null; }
  return 'stopped';
})()
''')
```

（同樣必須在**同一格**執行；換格後 `window` 就不同了，跑另一個 kernel 重啟即可。）

## 5. 要真正的即時偵測 → 回自己的電腦

Colab 這條路是抓拍。真即時請回本機終端機：

```powershell
cd C:\Users\princ\ivdc_handout
.\.venv-yolo\Scripts\python.exe src\05_yolo_detection.py --source 0 --imgsz 640
```

按 `q` 或關視窗結束。腳本會**自動掃描 0~6 並量亮度**，跳過黑屏虛擬裝置 ——
**不要手寫 `--source 2`**，索引每次開機都可能變。

## 6. Colab 上不用鏡頭的替代方案

| 需求 | 做法 |
|---|---|
| 證明「即時偵測」流程能跑 | `yolo_only.ipynb` 第 3 格（`bus.jpg` 合成 `clip.mp4`） |
| 用自己的影片 | 左側面板上傳 `.mp4` → `model.predict(source="my.mp4", stream=True)` |
| 量真實 FPS | 一定要回本機 |